# Mars MCD wind power generation

Model notes:

1. The Nature paper used hourly wind generation and 100 m wind speed.
2. `mcd_data.nc` provides `1m above surface` winds with `ut = 0, 0.5, ..., 23.5`, so it is 1 m, half-hourly U/V wind.
3. This notebook does **not** extrapolate to 100 m. It directly uses MCD 1 m wind.
4. The active model uses the paper reference turbine: Goldwind GW140-3MW, rotor diameter about **140 m**, rated power 3 MW.
5. Mars low-density air is handled by converting Mars wind speed to an Earth-density equivalent speed.

Output: hourly energy in `MWh` for one GW140-3MW turbine at each grid cell.

Paper turbine power-curve model:

$$
v_{eq}=v_{Mars}\left(\frac{\rho_{Mars}}{\rho_{Earth}}\right)^{1/3}
$$

Then `v_eq` is passed through the GW140-3MW turbine curve: cut-in speed `vc`, rated speed `vr`, cut-out speed `vf`, rated power `Pr`, and efficiency `eta`.

Reference turbine: `D≈140 m`, `Pr=3 MW`, `vc=2.5 m/s`, `vr=12 m/s`, `vf=20 m/s`, `eta=0.9`.

In [ ]:
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

# Run the notebook from its Mars folder; provide the external MCD file first.
project_dir = Path.cwd().parent
src_path = project_dir / "Data_MCD" / "mcd_data.nc"
if not src_path.is_file():
    raise FileNotFoundError(f"External MCD input is required: {src_path}")
ascii_dir = project_dir / "outputs"
ascii_dir.mkdir(parents=True, exist_ok=True)
mcd_path = src_path

out_path = ascii_dir / "mars_mcd_hourly_wind_generation_1m.nc"
final_out_path = project_dir / out_path.name

ds = xr.open_dataset(mcd_path, engine="h5netcdf")
ds

In [ ]:
# Confirm temporal resolution and data height
print("height:", ds.attrs.get("height"))
print("ut values:", ds["ut"].values[:6], "...", ds["ut"].values[-3:])
print("U dims:", ds["U"].dims, ds["U"].shape)

In [ ]:
# Utility-scale turbine model from the paper.
# Goldwind GW140-3MW: rotor diameter ~140 m, rated power 3 MW.

D = 140.0          # rotor diameter, m
Pr = 3.0           # rated power, MW
vc = 2.5           # cut-in wind speed, m/s, Earth-density turbine curve
vr = 12.0          # rated wind speed, m/s, Earth-density turbine curve
vf = 20.0          # cut-out wind speed, m/s, Earth-density turbine curve
eta = 0.90         # conversion efficiency

rho_mars = 0.020   # kg/m3, typical near-surface Mars atmospheric density
rho_earth = 1.225  # kg/m3, Earth sea-level air density
apply_density_correction = True
density_speed_factor = (rho_mars / rho_earth) ** (1 / 3) if apply_density_correction else 1.0

print("turbine: Goldwind GW140-3MW")
print("rotor diameter m:", D)
print("rated power MW:", Pr)
print("Mars-to-Earth equivalent wind-speed factor:", density_speed_factor)

def wind_power_mw(v_mars):
    v = v_mars * density_speed_factor
    p = xr.where((v >= vc) & (v < vr), Pr * (v**3 - vc**3) / (vr**3 - vc**3), 0)
    p = xr.where((v >= vr) & (v <= vf), Pr, p)
    p = xr.where(v > vf, 0, p)
    return (eta * p).astype("float32")

In [ ]:
# MCD U/V are 1 m winds. Use them directly, no 100 m extrapolation.
wind_speed_1m = np.hypot(ds["U"], ds["V"]).astype("float32")
power_mw_30min = wind_power_mw(wind_speed_1m)

# Convert 48 half-hour power values into 24 hourly energy values.
# MWh for each hour = P(t)*0.5h + P(t+0.5)*0.5h
power_pair = power_mw_30min.coarsen(ut=2, boundary="trim")
energy_mwh_hourly = (power_pair.sum() * 0.5).rename("wind_generation_MWh")
energy_mwh_hourly = energy_mwh_hourly.assign_coords(ut=np.arange(24)).rename({"ut": "hour"})

energy_mwh_hourly.attrs.update({
    "long_name": "Hourly wind generation using MCD 1 m wind speed",
    "units": "MWh per grid cell per hour for one GW140-3MW turbine",
    "source_wind_height": ds.attrs.get("height", "1m above surface"),
    "method": "GW140-3MW turbine power curve with Mars-to-Earth density equivalent wind-speed correction; no 100 m extrapolation",
    "turbine": "Goldwind GW140-3MW",
    "rotor_diameter_m": D,
    "vc_m_s": vc,
    "vr_m_s": vr,
    "vf_m_s": vf,
    "rated_power_MW": Pr,
    "efficiency": eta,
    "rho_mars_kg_m3": rho_mars,
    "rho_earth_kg_m3": rho_earth,
    "density_speed_factor": density_speed_factor,
})

energy_mwh_hourly

In [ ]:
out = energy_mwh_hourly.to_dataset()
out.to_netcdf(out_path)
shutil.copy2(out_path, final_out_path)
print("saved ascii working file:", out_path.resolve())
print("saved final file:", final_out_path.resolve())

In [ ]:
# InSight landing site: 4.5024 N, 135.6234 E. Use the nearest MCD grid cell.
insight_lat = 4.5024
insight_lon = 135.6234

insight_hourly = out.wind_generation_MWh.sel(
    longitude=insight_lon,
    latitude=insight_lat,
    method="nearest",
)
grid_lon = float(insight_hourly.longitude)
grid_lat = float(insight_hourly.latitude)

# Daily / seasonal-step totals for the InSight nearest grid cell.
daily_mwh = insight_hourly.sum(dim="hour").rename("daily_generation_MWh")
total_mwh = float(daily_mwh.sum())
total_twh = total_mwh / 1e6

# China comparison: 2024 electricity production is about 10,086.88 TWh, so average 1 h is annual/8760.
china_annual_twh = 10086.88
world_annual_twh = 30937.0
china_avg_1h_twh = china_annual_twh / 8760
world_avg_1h_twh = world_annual_twh / 8760

china_equiv_hours = total_twh / china_avg_1h_twh
world_equiv_hours = total_twh / world_avg_1h_twh

summary_csv = project_dir / "mars_mcd_insight_daily_wind_generation_1m.csv"
fig_path = project_dir / "mars_mcd_insight_daily_wind_generation_1m.png"

daily_df = daily_mwh.to_dataframe().reset_index()
daily_df.insert(0, "mcd_step", np.arange(1, len(daily_df) + 1))
daily_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")

plt.figure(figsize=(14, 4))
plt.bar(daily_df["mcd_step"], daily_df["daily_generation_MWh"], width=1.0)
plt.xlabel("MCD seasonal step (Ls step)")
plt.ylabel("Generation (MWh)")
plt.title(f"Daily wind generation near InSight ({grid_lat:.2f}N, {grid_lon:.2f}E)")
plt.tight_layout()
plt.savefig(fig_path, dpi=200)
plt.show()

global_total_twh = float(out.wind_generation_MWh.sum() / 1e6)

print(f"InSight target: lat={insight_lat}, lon={insight_lon}")
print(f"nearest MCD grid: lat={grid_lat}, lon={grid_lon}")
print(f"InSight-grid total generation: {total_mwh:.6g} MWh = {total_twh:.6g} TWh")
print(f"Global-grid total generation for comparison: {global_total_twh:.6g} TWh")
print(f"China average 1 hour electricity: {china_avg_1h_twh:.3f} TWh")
print(f"Equivalent to China average electricity use for: {china_equiv_hours:.6g} hours")
print(f"Equivalent to world average electricity use for: {world_equiv_hours:.6g} hours")
print("saved daily csv:", summary_csv)
print("saved figure:", fig_path)

In [ ]:
# Wind-speed diagnostic at the same InSight nearest grid cell.
insight_wind = wind_speed_1m.sel(
    longitude=insight_lon,
    latitude=insight_lat,
    method="nearest",
)

wind_daily = xr.Dataset({
    "wind_mean_m_s": insight_wind.mean(dim="ut"),
    "wind_max_m_s": insight_wind.max(dim="ut"),
})

wind_csv = project_dir / "mars_mcd_insight_daily_wind_speed_1m.csv"
wind_fig = project_dir / "mars_mcd_insight_daily_wind_speed_1m.png"
wind_df = wind_daily.to_dataframe().reset_index()
wind_df.insert(0, "mcd_step", np.arange(1, len(wind_df) + 1))
wind_df.to_csv(wind_csv, index=False, encoding="utf-8-sig")

raw_cut_in = vc / density_speed_factor

plt.figure(figsize=(14, 4))
plt.plot(wind_df["mcd_step"], wind_df["wind_mean_m_s"], label="daily mean 1 m wind")
plt.plot(wind_df["mcd_step"], wind_df["wind_max_m_s"], label="daily max 1 m wind", alpha=0.8)
plt.axhline(raw_cut_in, color="r", linestyle="--", label=f"raw Mars cut-in: {raw_cut_in:.2f} m/s")
plt.xlabel("MCD seasonal step (Ls step)")
plt.ylabel("Wind speed (m/s)")
plt.title(f"1 m wind speed near InSight ({grid_lat:.2f}N, {grid_lon:.2f}E)")
plt.legend()
plt.tight_layout()
plt.savefig(wind_fig, dpi=200)
plt.show()

season_bins = [0, 90, 180, 270, 360.1]
season_labels = ["Ls 0-90 N spring", "Ls 90-180 N summer", "Ls 180-270 N autumn", "Ls 270-360 N winter"]
wind_df["season"] = pd.cut(wind_df["Ls"], bins=season_bins, labels=season_labels, include_lowest=True, right=False)
print(wind_df.groupby("season", observed=False)[["wind_mean_m_s", "wind_max_m_s"]].agg(["mean", "max"]))
print("saved wind csv:", wind_csv)
print("saved wind figure:", wind_fig)

In [ ]:
# Quick checks
print(out)
print("wind speed min / mean / max m/s:",
      float(wind_speed_1m.min()),
      float(wind_speed_1m.mean()),
      float(wind_speed_1m.max()))
print("exact zero wind-speed values:", int((wind_speed_1m == 0).sum()))
print("min / mean / max MWh:",
      float(out.wind_generation_MWh.min()),
      float(out.wind_generation_MWh.mean()),
      float(out.wind_generation_MWh.max()))

In [ ]:
# Wind speed -> power curve for the GW140-3MW model with density correction.
v_mars = xr.DataArray(np.linspace(0, 100, 1001), dims="v_mars", name="v_mars")
p_curve = wind_power_mw(v_mars)

cut_in_raw = vc / density_speed_factor
rated_raw = vr / density_speed_factor
cut_out_raw = vf / density_speed_factor

curve_csv = project_dir / "mars_wind_speed_power_curve.csv"
curve_fig = project_dir / "mars_wind_speed_power_curve.png"

curve_df = pd.DataFrame({
    "mars_wind_speed_m_s": v_mars.values,
    "earth_equivalent_wind_speed_m_s": v_mars.values * density_speed_factor,
    "rotor_diameter_m": D,
    "power_MW": p_curve.values,
})
curve_df.to_csv(curve_csv, index=False, encoding="utf-8-sig")

plt.figure(figsize=(8, 4.5))
plt.plot(curve_df["mars_wind_speed_m_s"], curve_df["power_MW"], lw=2)
plt.axvline(cut_in_raw, color="gray", ls="--", label=f"cut-in {cut_in_raw:.1f} m/s")
plt.axvline(rated_raw, color="tab:orange", ls="--", label=f"rated {rated_raw:.1f} m/s")
plt.axvline(cut_out_raw, color="tab:red", ls="--", label=f"cut-out {cut_out_raw:.1f} m/s")
plt.xlabel("Mars 1 m wind speed (m/s)")
plt.ylabel("Power (MW)")
plt.title("Wind speed to power curve, GW140-3MW with density correction")
plt.ylim(-0.00001, Pr * eta * 1.08)
plt.legend()
plt.tight_layout()
plt.savefig(curve_fig, dpi=200)
plt.show()

print("raw Mars wind speed thresholds after density correction:")
print(f"cut-in:  {cut_in_raw:.2f} m/s")
print(f"rated:   {rated_raw:.2f} m/s")
print(f"cut-out: {cut_out_raw:.2f} m/s")
print("saved curve csv:", curve_csv)
print("saved curve figure:", curve_fig)